In [4]:
import pandas as pd

# Загружаем данные (предположим, твой файл называется data.csv)
# Если ты просто вставил строку, создадим DataFrame для теста:
df = pd.read_csv('../data/sensors.csv', sep=';') # или другой разделитель

# Список каналов, которые нам реально нужны для визуализации
essential_channels = [
    'Time',           # Время от начала лога
    'DistanceLap',    # Позиция на трассе (в метрах)
    'CarSpeed',       # Скорость для спидометра
    'RPM',            # Обороты для тахометра
    'Gear',           # Передача
    'rPedal',         # Газ (0-100%)
    'pBrakeF',        # Тормоз передний (давление)
    'aSteering',      # Угол руля
    'AccX_Corr',      # Продольное ускорение
    'AccY_Corr'       # Боковое ускорение
]

# Оставляем только нужное
df_sim = df[essential_channels].copy()

# Быстрая проверка: есть ли пустые значения
print(f"Загружено строк: {len(df_sim)}")
df_sim['DistanceLap'].max()
df_sim

Загружено строк: 19367


,Time,DistanceLap,CarSpeed,RPM,Gear,rPedal,pBrakeF,aSteering,AccX_Corr,AccY_Corr
0,0:000,3820,"144,5",5289,4,"101,4","0,3","-2,7","1,063","-0,726"
1,0:005,3820,"144,5",5302,4,"101,4","0,2","-2,6","0,919","0,524"
2,0:010,3820,"144,6",5303,4,"101,4","0,2","-2,6","0,919","0,524"
3,0:015,3821,"144,7",5298,4,"101,4","0,2","-2,6","0,945","0,368"
4,0:020,3821,"144,7",5292,4,"101,4","0,2","-2,6","0,945","0,368"
...,...,...,...,...,...,...,...,...,...,...
19362,96:810,3827,"142,9",5232,4,"101,4","0,2","-2,3","0,653","0,285"
19363,96:815,3827,"143,0",5228,4,"101,5","0,2","-2,0","-0,222","0,847"
19364,96:820,3827,"143,0",5240,4,"101,5","0,2","-2,0","-0,222","0,847"
19365,96:825,3828,"143,1",5224,4,"101,6","0,3","-1,9","-0,037","-0,523"


In [5]:
import pandas as pd

# 1. Сначала превращаем всё в строки, чтобы метод .str сработал везде
df_sim = df_sim.astype(str)

# 2. Массовая замена запятых на точки во всей таблице
for col in df_sim.columns:
    df_sim[col] = df_sim[col].str.replace(',', '.')

# 3. Специальная функция для времени (обрабатываем формат ММ:СС.ms)
def fix_time(t_str):
    try:
        if ':' in t_str:
            parts = t_str.split(':')
            # Если формат 96:815 (минуты:секунды.доли)
            return float(parts[0]) * 60 + float(parts[1])
        return float(t_str)
    except:
        return 0.0

# 4. Применяем конвертацию к колонкам
df_sim['Time'] = df_sim['Time'].apply(fix_time)

# 5. Все остальные колонки просто в числа
numeric_cols = ['DistanceLap', 'CarSpeed', 'RPM', 'Gear', 'rPedal', 'pBrakeF', 'aSteering', 'AccX_Corr', 'AccY_Corr']
for col in numeric_cols:
    df_sim[col] = pd.to_numeric(df_sim[col], errors='coerce')

# 6. Заполняем пустоты
df_sim = df_sim.ffill().fillna(0)

# 7. Финальный ПРОВЕРОЧНЫЙ ВЫВОД
print("ПРОВЕРКА ТИПОВ:")
print(df_sim.dtypes)
print("\nПЕРВЫЕ СТРОКИ (убедись, что нет запятых):")
display(df_sim.head())

ПРОВЕРКА ТИПОВ:
Time           float64
DistanceLap      int64
CarSpeed       float64
RPM              int64
Gear             int64
rPedal         float64
pBrakeF        float64
aSteering      float64
AccX_Corr      float64
AccY_Corr      float64
dtype: object

ПЕРВЫЕ СТРОКИ (убедись, что нет запятых):


,Time,DistanceLap,CarSpeed,RPM,Gear,rPedal,pBrakeF,aSteering,AccX_Corr,AccY_Corr
0,0.0,3820,144.5,5289,4,101.4,0.3,-2.7,1.063,-0.726
1,5.0,3820,144.5,5302,4,101.4,0.2,-2.6,0.919,0.524
2,10.0,3820,144.6,5303,4,101.4,0.2,-2.6,0.919,0.524
3,15.0,3821,144.7,5298,4,101.4,0.2,-2.6,0.945,0.368
4,20.0,3821,144.7,5292,4,101.4,0.2,-2.6,0.945,0.368


In [13]:
import json
import math

# --- НАСТРОЙКИ ВИЗУАЛА ---
SCREEN_WIDTH = 400
SCREEN_HEIGHT = 320
TRACK_FILE = '../data/track.geojson'

def load_track(filename):
    with open(filename, 'r') as f:
        data = json.load(f)
    return data['features'][0]['geometry']['coordinates']

def scale_coords(coords, width, height, padding=40):
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    min_lon, max_lon = min(lons), max(lons)
    min_lat, max_lat = min(lats), max(lats)
    scale = min((width - 2*padding) / (max_lon - min_lon), (height - 2*padding) / (max_lat - min_lat))
    return [(padding + (lon - min_lon) * scale, height - padding - (lat - min_lat) * scale) for lon, lat in coords]

def smooth_chaikin(points, iterations=4):
    for _ in range(iterations):
        new_points = []
        for i in range(len(points)):
            p0 = points[i]; p1 = points[(i + 1) % len(points)]
            new_points.append((0.75 * p0[0] + 0.25 * p1[0], 0.75 * p0[1] + 0.25 * p1[1]))
            new_points.append((0.25 * p0[0] + 0.75 * p1[0], 0.25 * p0[1] + 0.75 * p1[1]))
        points = new_points
    return points

def equalize_track(points, step_size=1.0):
    new_points = [points[0]]
    temp_points = points + [points[0]]
    leftover = 0.0
    for i in range(len(temp_points) - 1):
        p1 = temp_points[i]; p2 = temp_points[i+1]
        dist = math.hypot(p2[0]-p1[0], p2[1]-p1[1])
        if dist == 0: continue
        dir_x, dir_y = (p2[0]-p1[0])/dist, (p2[1]-p1[1])/dist
        current_pos = leftover
        while current_pos < dist:
            new_points.append((p1[0] + dir_x * current_pos, p1[1] + dir_y * current_pos))
            current_pos += step_size
        leftover = current_pos - dist
    return new_points

# --- ГЕНЕРАЦИЯ ГЕОМЕТРИИ ---
raw_coords = load_track(TRACK_FILE)
scaled_base = scale_coords(raw_coords, SCREEN_WIDTH, SCREEN_HEIGHT)
smoothed = smooth_chaikin(scaled_base, iterations=4)
# final_points — это наша "линейка" в пикселях
final_points = equalize_track(smoothed, step_size=0.5) 

print(f"Трасса готова. Длина в точках (пикселях): {len(final_points)}")

Трасса готова. Длина в точках (пикселях): 1759


In [14]:
# Максимальная дистанция из твоего лога (3828 м)
max_lap_dist = df_sim['DistanceLap'].max()
num_track_points = len(final_points)

# Создаем индекс: привязываем метры лога к индексам массива final_points
# Используем коэффициент: (текущий_метр / макс_метр) * кол-во_точек_массива
df_sim['track_index'] = (df_sim['DistanceLap'] / max_lap_dist * (num_track_points - 1)).astype(int)
df_sim['track_index'] = df_sim['track_index'].clip(0, num_track_points - 1)

print("Телеметрия синхронизирована с геометрией трассы.")

Телеметрия синхронизирована с геометрией трассы.


In [15]:
df_sim

,Time,DistanceLap,CarSpeed,RPM,Gear,rPedal,pBrakeF,aSteering,AccX_Corr,AccY_Corr,track_index
0,0.0,3820,144.5,5289,4,101.4,0.3,-2.7,1.063,-0.726,1754
1,5.0,3820,144.5,5302,4,101.4,0.2,-2.6,0.919,0.524,1754
2,10.0,3820,144.6,5303,4,101.4,0.2,-2.6,0.919,0.524,1754
3,15.0,3821,144.7,5298,4,101.4,0.2,-2.6,0.945,0.368,1754
4,20.0,3821,144.7,5292,4,101.4,0.2,-2.6,0.945,0.368,1754
...,...,...,...,...,...,...,...,...,...,...,...
19362,6570.0,3827,142.9,5232,4,101.4,0.2,-2.3,0.653,0.285,1757
19363,6575.0,3827,143.0,5228,4,101.5,0.2,-2.0,-0.222,0.847,1757
19364,6580.0,3827,143.0,5240,4,101.5,0.2,-2.0,-0.222,0.847,1757
19365,6585.0,3828,143.1,5224,4,101.6,0.3,-1.9,-0.037,-0.523,1758


In [20]:
import pygame
# --- ИНИЦИАЛИЗАЦИЯ ---
pygame.init()
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

# Используем системные шрифты (убедись, что они есть, или замени на стандартный)
try:
    font = pygame.font.SysFont("arial", 18)
    font_bold = pygame.font.SysFont("arial", 24, bold=True)
except:
    font = pygame.font.Font(None, 24)
    font_bold = pygame.font.Font(None, 32)

log_step = 0
running = True

# Разворачиваем список точек, чтобы изменить направление движения
final_points = final_points[::-1]

while running:
    screen.fill((30, 30, 30))
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 1. ПОЛУЧЕНИЕ ДАННЫХ
    row = df_sim.iloc[log_step]
    
    # Принудительно конвертируем в числа (на случай, если остались строки)
    speed = float(row['CarSpeed'])
    rpm   = float(row['RPM'])
    gear  = int(row['Gear'])
    pedal = float(row['rPedal'])
    brake = float(row['pBrakeF'])
    dist  = float(row['DistanceLap'])
    time_val = float(row['Time'])
    t_idx = int(row['track_index'])

    # 2. ОТРИСОВКА ТРАССЫ
    pygame.draw.aalines(screen, (100, 100, 100), True, final_points)

    # 3. ОТРИСОВКА МАШИНЫ
    curr_pos = final_points[t_idx]
    # Цвет: красный если тормоз > 5, зеленый если газ > 20
    color = (255, 255, 0)
    if brake > 5: color = (255, 50, 50)
    elif pedal > 20: color = (50, 255, 50)
    
    pygame.draw.circle(screen, color, (int(curr_pos[0]), int(curr_pos[1])), 7)

    # 4. ВЫВОД ТЕКСТА (Интерфейс)
    # Скорость
    txt_speed = font_bold.render(f"Speed: {speed:.1f} km/h", True, (255, 255, 255))
    screen.blit(txt_speed, (20, 20))
    
    # Передача и обороты
    txt_gear = font_bold.render(f"Gear: {gear}", True, (100, 200, 255))
    screen.blit(txt_gear, (20, 50))
    
    txt_rpm = font.render(f"RPM: {int(rpm)}", True, (255, 255, 255))
    screen.blit(txt_rpm, (20, 80))

    # Визуализация педалей
    # ГАЗ (зеленый)
    pygame.draw.rect(screen, (50, 50, 50), (20, 120, 100, 10)) # фон
    pygame.draw.rect(screen, (0, 255, 0), (20, 120, int(min(pedal, 100)), 10)) 
    
    # ТОРМОЗ (красный)
    pygame.draw.rect(screen, (50, 50, 50), (20, 140, 100, 10)) # фон
    pygame.draw.rect(screen, (255, 0, 0), (20, 140, int(min(brake, 100)), 10))

    # Время и дистанция (справа)
    txt_time = font.render(f"Time: {time_val:.2f} s", True, (200, 200, 200))
    screen.blit(txt_time, (SCREEN_WIDTH - 150, 20))
    
    txt_dist = font.render(f"Dist: {int(dist)} m", True, (200, 200, 200))
    screen.blit(txt_dist, (SCREEN_WIDTH - 150, 45))

    # 5. ОБНОВЛЕНИЕ
    log_step += 1
    if log_step >= len(df_sim):
        log_step = 0

    pygame.display.flip()
    clock.tick(200) 

pygame.quit()

In [3]:
%pip install opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 6.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import numpy as np

# Загружаем изображение
img = cv2.imread('../data/sample_frame.jpg')
h, w = img.shape[:2]

# --- ШАГ 1: Выбираем 4 точки на исходном изображении (src_points) ---
# Эти координаты ты должен подобрать под свою камеру вручную.
# Это пример для случая, когда дорога впереди занимает центр кадра.
# Лучше всего выбирать их интерактивно (см. следующий блок кода).
src_points = np.float32([[275, 285], [430, 280], [350, 340], [85, 345]])

# --- ШАГ 2: Задаем соответствующие точки на виде сверху (dst_points) ---
# Мы хотим получить изображение сверху размером 500x500.
# Левый нижний угол (ближний к машине) станет левым нижним на BirdView.
# Дальние точки станут верхними углами.

width_bird = 1080
height_bird = 720
shara = 300

dst_points = np.float32([
       
    [706, shara + 120],       
    [769, shara + 120],
               [769, shara],
                          [706, shara],          
])

# --- ШАГ 3: Вычисляем матрицу гомографии ---
# getPerspectiveTransform работает строго для 4 точек, findHomography - более общий случай.
H, status = cv2.findHomography(src_points, dst_points)
# Или так (если точки точно соответствуют друг другу):
# H = cv2.getPerspectiveTransform(src_points, dst_points)

# --- ШАГ 4: Применяем преобразование ---
img_bird = cv2.warpPerspective(img, H, (width_bird, height_bird))

# Показываем результат
# cv2.imshow('Original', img)
cv2.imshow('Bird Eye View', img_bird)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [28]:
import cv2
import numpy as np

# Загружаем изображение
img = cv2.imread('../data/sample_frame.jpg')
h, w = img.shape[:2]

# --- ШАГ 1: Выбираем 4 точки на исходном изображении (src_points) ---
# Эти координаты ты должен подобрать под свою камеру вручную.
# Это пример для случая, когда дорога впереди занимает центр кадра.
# Лучше всего выбирать их интерактивно (см. следующий блок кода).
src_points = np.float32([[85, 345], [350, 340], [275, 285], [430, 280]])

# --- ШАГ 2: Задаем соответствующие точки на виде сверху (dst_points) ---
# Мы хотим получить изображение сверху размером 500x500.
# Левый нижний угол (ближний к машине) станет левым нижним на BirdView.
# Дальние точки станут верхними углами.

width_bird = 1080
height_bird = 5000
shara = 400

dst_points = np.float32([
    [706, shara + 120],                    # Левый верхний (src дальний левый)
    [769, shara + 120] ,           # Правый верхний (src дальний правый)
    [706, shara],          # Левый нижний (src ближний левый)
    [769, shara], # Правый нижний (src ближний правый)
])

# --- ШАГ 3: Вычисляем матрицу гомографии ---
# getPerspectiveTransform работает строго для 4 точек, findHomography - более общий случай.
H, status = cv2.findHomography(src_points, dst_points)
# Или так (если точки точно соответствуют друг другу):
# H = cv2.getPerspectiveTransform(src_points, dst_points)

# --- ШАГ 4: Применяем преобразование ---
img_bird = cv2.warpPerspective(img, H, (width_bird, height_bird))

# Показываем результат
# cv2.imshow('Original', img)
cv2.imshow('Bird Eye View', img_bird)
cv2.waitKey(0)
cv2.destroyAllWindows()